# Topic Modeling of Recent Academic Publications using KeyBERT + KMeans
## Discovering Hot Research Topics from 2020 to 2025

In [ ]:
!pip install keybert
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan
!pip install plotly

In [ ]:
!pip install -qq numpy==1.26.4 gensim
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/MachineLearning/processed.csv")
print(df.shape)
df.head()

In [ ]:
df = df.dropna(subset=['cleaned_abstract'])
documents = df['cleaned_abstract'].tolist()
years = df["year"].astype(str).tolist()
print(f"Total documents: {len(documents)}")
print("First document sample:")
print(documents[0][:300], "...")

## 2. Modeling

In [ ]:
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

model_1 = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
kw_model_1 = KeyBERT(model=model_1)

#model_2 = SentenceTransformer('all-mpnet-base-v2', device='cuda')
#kw_model_2 = KeyBERT(model=model_2)

In [ ]:
sample_doc = documents[0]

sample_keywords_1 = kw_model_1.extract_keywords(sample_doc,
                                                keyphrase_ngram_range=(1, 3),
                                                stop_words='english',
                                                top_n=10)

#sample_keywords_2= kw_model_2.extract_keywords(sample_doc,
                                               #keyphrase_ngram_range=(1, 3),
                                               #stop_words='english',
                                               #top_n=10)

print("all-MiniLM-L6-v2:")
print(sample_keywords_1)

#print("\nall-mpnet-base-v2:")
#print(sample_keywords_2)

### 2.1 Keyword Extracting

In [ ]:
from tqdm.notebook import tqdm
import pickle

keywords_1 = [kw_model_1.extract_keywords(doc,
                                keyphrase_ngram_range=(1, 2),
                                top_n=15,
                                stop_words='english')
    for doc in tqdm(documents, desc="Extracting keywords with model 1")
]

save_path_1 = "/content/drive/MyDrive/keyword_results_model1.pkl"

with open(save_path_1, "wb") as f:
    pickle.dump(keywords_1, f)

print("Saved.")

In [ ]:
import pickle

load_path_1 = "/content/drive/MyDrive/keyword_results_model1.pkl"

with open(load_path_1, "rb") as f:
    keywords_1 = pickle.load(f)

### 2.2. KMeans Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

keywords_only_1 = [' '.join([kw for kw, _ in kws]) for kws in keywords_1]

embeddings_1 = model_1.encode(keywords_only_1, show_progress_bar=True)

n_clusters = 100
kmeans_1 = KMeans(n_clusters=n_clusters, random_state=42).fit(embeddings_1)
tsne_1 = TSNE(n_components=2, random_state=42).fit_transform(embeddings_1)

## 3. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_clusters(tsne_result, labels, title):
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x=tsne_result[:,0], y=tsne_result[:,1], hue=labels, palette='tab20', s=50, legend=False)
    plt.title(title)
    plt.xlabel('TSNE-1')
    plt.ylabel('TSNE-2')
    plt.tight_layout()
    plt.show()

plot_clusters(tsne_1, kmeans_1.labels_, 'Clusters using all-MiniLM-L6-v2 Embedding')

#### Assigning Topic Labels

In [ ]:
import numpy as np
from collections import Counter, defaultdict

def assign_topic_labels(embeddings, kmeans_labels, keywords_list, n_top_words=5):
    topic_labels = {}
    cluster_centers = kmeans_1.cluster_centers_

    for cluster_id in range(n_clusters):
        cluster_keywords = []
        cluster_indices = np.where(kmeans_labels == cluster_id)[0]

        for idx in cluster_indices:
            cluster_keywords.extend([kw for kw, score in keywords_list[idx]])

        keyword_counts = Counter(cluster_keywords)
        top_keywords = [kw for kw, count in keyword_counts.most_common(n_top_words)]

        topic_labels[cluster_id] = f"Topic_{cluster_id}: {', '.join(top_keywords[:5])}"

    return topic_labels

topic_labels = assign_topic_labels(embeddings_1, kmeans_1.labels_, keywords_1)

for topic_id, label in topic_labels.items():
    print(f"{label}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def create_topic_similarity_matrix(cluster_centers):
    similarity_matrix = cosine_similarity(cluster_centers)
    return similarity_matrix

similarity_matrix = create_topic_similarity_matrix(kmeans_1.cluster_centers_)

plt.figure(figsize=(12, 10))
sns.heatmap(similarity_matrix,
            annot=False,
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=0.1)
plt.title('Topic-Topic Cosine Similarity Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

cluster_centers = kmeans_1.cluster_centers_
print(f"Cluster centers shape: {cluster_centers.shape}")

center_linkage = linkage(cluster_centers, method='ward')

plt.figure(figsize=(10, 20))
dendrogram(center_linkage,
         labels=[f"Topic {i}" for i in range(len(cluster_centers))],
         orientation='right',
         leaf_font_size=8)
plt.title('Hierarchical Clustering of K-means Centers', fontsize=16)
plt.xlabel('Distance', fontsize=12)
plt.ylabel('Topics', fontsize=12)
plt.tight_layout()
plt.show()

### Topics over Time - Growing Topics from 2020 to 2025

In [ ]:
def calculate_topic_trends(labels, years):
    topic_year_counts = defaultdict(lambda: defaultdict(int))

    for label, year in zip(labels, years):
        topic_year_counts[label][year] += 1

    return topic_year_counts

topic_trends = calculate_topic_trends(kmeans_1.labels_, years)

trend_data = []
for topic_id, year_counts in topic_trends.items():
    for year, count in year_counts.items():
        trend_data.append({
            'topic': topic_id,
            'year': int(year),
            'count': count,
            'topic_label': topic_labels[topic_id]
        })

trend_df = pd.DataFrame(trend_data)

In [ ]:
def find_trending_topics(trend_df, top_n=10):
    recent_years = trend_df['year'].max() - 4

    topic_growth = {}
    for topic_id in trend_df['topic'].unique():
        topic_data = trend_df[trend_df['topic'] == topic_id]

        recent_count = topic_data[topic_data['year'] >= recent_years]['count'].sum()
        old_count = topic_data[topic_data['year'] < recent_years]['count'].sum()

        if old_count > 0:
            growth_rate = (recent_count - old_count) / old_count * 100
        else:
            growth_rate = recent_count * 100

        topic_growth[topic_id] = growth_rate

    trending_topics = sorted(topic_growth.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return trending_topics

trending_topics = find_trending_topics(trend_df)
trending_topic_ids = [topic_id for topic_id, growth in trending_topics]

print("Growing topics from 2020 to 2025:")
for topic_id, growth in trending_topics:
    print(f"{topic_labels[topic_id]}: {growth:.1f}% growth")

In [ ]:
plt.figure(figsize=(15, 10))

topic_data_dict = {}
for topic_id in trending_topic_ids:
   topic_data = trend_df[trend_df['topic'] == topic_id]
   topic_yearly = topic_data.groupby('year')['count'].sum().reset_index()
   topic_data_dict[topic_id] = topic_yearly

for topic_id, yearly_data in topic_data_dict.items():
   topic_label = topic_labels[topic_id].split(': ')[1].split(', ')[:5]
   short_label = ', '.join(topic_label)

   plt.plot(yearly_data['year'], yearly_data['count'],
            marker='o', linewidth=2, label=short_label)

plt.title('En Hızlı Büyüyen Araştırma Konuları', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Document Count', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### WordCloud - Growing Topics from 2020 to 2025

In [ ]:
from wordcloud import WordCloud

def create_topic_wordclouds(trending_topic_ids, keywords_list, kmeans_labels, topic_labels):

    for i, topic_id in enumerate(trending_topic_ids):
        topic_keywords = []
        topic_indices = np.where(kmeans_labels == topic_id)[0]

        for idx in topic_indices:
            for kw, score in keywords_list[idx]:
                topic_keywords.extend([kw] * int(score * 100))

        if topic_keywords:
            plt.figure(figsize=(12, 8))

            wordcloud = WordCloud(width=800, height=600,
                                background_color='white',
                                colormap='viridis',
                                max_words=100,
                                relative_scaling=0.5,
                                min_font_size=10).generate(' '.join(topic_keywords))

            plt.imshow(wordcloud, interpolation='bilinear')

            topic_label = topic_labels[topic_id].split(': ')[1]
            plt.title(f'Topic {topic_id}', fontsize=16, pad=20)
            plt.axis('off')
            plt.tight_layout()
            print(f"Topic {i+1}")
            plt.show()

create_topic_wordclouds(trending_topic_ids, keywords_1, kmeans_1.labels_, topic_labels)

## Growing Topics from 2020 to 2025

| Topic ID | Description                                                |
|----------|------------------------------------------------------------|
| 22       | Diffusion-Based Generative Models & Denoising Techniques  |
| 47       | Vision-Language Architectures & Cross-Modal AI            |
| 37       | Multimodal Learning Strategies & Data Fusion               |
| 83       | Text-to-Image Synthesis & Visual Captioning                |
| 50       | Scalable Language Models & Pretraining Paradigms           |
| 11       | General AI Systems & Foundational Model Design             |
| 70       | Style Transfer Methods & Creative Image Manipulation       |
| 79       | Steganography Techniques & Digital Watermarking Systems    |
| 45       | Structured Reasoning in LLMs & Knowledge Representation    |
| 5        | Neural Rendering, Illumination & Image Enhancement         |

## 4. Performance Metrics

### 4.1. Coherence Scores

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from collections import Counter

tokenized_docs = [doc.split() for doc in documents if isinstance(doc, str)]

dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

topics_words = []
for cluster_id in range(n_clusters):
    cluster_keywords = []
    cluster_indices = np.where(kmeans_1.labels_ == cluster_id)[0]

    for idx in cluster_indices:
        for kw, score in keywords_1[idx]:
            if isinstance(kw, str):
                if " " not in kw:
                    cluster_keywords.append(kw)
                else:
                    cluster_keywords.extend(kw.split())

    if cluster_keywords:
        word_counts = Counter(cluster_keywords)
        top_words = [word for word, count in word_counts.most_common(10)]
        clean_words = [word for word in top_words if word in dictionary.token2id]
        if clean_words:
            topics_words.append(clean_words)

coherence_cv = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v').get_coherence()
coherence_umass = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='u_mass').get_coherence()
coherence_npmi = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi').get_coherence()
coherence_uci = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_uci').get_coherence()

print(f"C_v Coherence: {coherence_cv:.4f}")
print(f"U_Mass Coherence: {coherence_umass:.4f}")
print(f"NPMI Coherence: {coherence_npmi:.4f}")
print(f"UCI Coherence: {coherence_uci:.4f}")

### 4.2. PUW

In [ ]:
all_words = [word for topic in topics_words for word in topic]
unique_words = set(all_words)
puw = len(unique_words) / len(all_words)
print(f"PUW (Proportion of Unique Words): {puw:.4f}")

### 4.3. Avg. Jaccard Similarity

In [ ]:
from itertools import combinations

def jaccard_similarity(set1, set2):
   return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for t1, t2 in combinations(topics_words, 2):
   jaccard_scores.append(jaccard_similarity(set(t1), set(t2)))

avg_jaccard = sum(jaccard_scores) / len(jaccard_scores)
print(f"Average Jaccard Similarity between topics: {avg_jaccard:.4f}")

### 4.4. Clustering Metrics

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

silhouette_avg = silhouette_score(embeddings_1, kmeans_1.labels_)
print(f"Silhouette Score: {silhouette_avg:.4f}")

calinski_score = calinski_harabasz_score(embeddings_1, kmeans_1.labels_)
print(f"Calinski-Harabasz Index: {calinski_score:.4f}")

davies_bouldin = davies_bouldin_score(embeddings_1, kmeans_1.labels_)
print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")

inertia = kmeans_1.inertia_
print(f"Inertia (WCSS): {inertia:.4f}")

unique, counts = np.unique(kmeans_1.labels_, return_counts=True)
print(f"Clusters: {len(unique)}")
print(f"Largest cluster: {max(counts)}")
print(f"Smallest cluster: {min(counts)}")
print(f"Avg. cluster size: {np.mean(counts):.1f}")